# Phase 2 Baseline (HF API +  Qwen)

Clean Colab notebook:
- Uses Hugging Face API token (`HF_TOKEN`)
- Uses Mistral model via chat/completions route
- Runs two baselines (ZeroShot, Prompted)


In [1]:
!pip -q install openai scikit-learn rouge-score nltk pandas


In [2]:
import os
import json
import re
import time
import glob
from pathlib import Path

import pandas as pd
import nltk
from huggingface_hub import InferenceClient
from sklearn.metrics import accuracy_score, f1_score, classification_report
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

nltk.download('punkt', quiet=True)
print('Imports ready')


Imports ready


In [3]:
# Path setup
candidates = ["/content/test.jsonl", "/test.jsonl"]
test_matches = [p for p in candidates if Path(p).exists()]
if not test_matches:
    test_matches = glob.glob('/kaggle/input/**/test.jsonl', recursive=True)

if not test_matches:
    raise FileNotFoundError('test.jsonl not found in /content or kaggle input')

TEST_FILE = test_matches[0]
DATA_DIR = str(Path(TEST_FILE).parent)
TRAIN_FILE = str(Path(DATA_DIR) / 'train.jsonl')
VAL_FILE = str(Path(DATA_DIR) / 'val.jsonl')
MASTER_FILE = str(Path(DATA_DIR) / 'master_with_splits.json')
if not Path(MASTER_FILE).exists():
    MASTER_FILE = '/content/master_with_splits.json' if Path('/content/master_with_splits.json').exists() else None

OUTPUT_DIR = '/content/phase2_baseline_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('TEST_FILE  :', TEST_FILE)
print('TRAIN_FILE :', TRAIN_FILE, '| exists:', Path(TRAIN_FILE).exists())
print('VAL_FILE   :', VAL_FILE, '| exists:', Path(VAL_FILE).exists())
print('MASTER_FILE:', MASTER_FILE)
print('OUTPUT_DIR :', OUTPUT_DIR)


TEST_FILE  : /content/test.jsonl
TRAIN_FILE : /content/train.jsonl | exists: True
VAL_FILE   : /content/val.jsonl | exists: True
MASTER_FILE: None
OUTPUT_DIR : /content/phase2_baseline_outputs


In [9]:
from openai import OpenAI
import os, time

client = OpenAI(
    base_url="https://api.fireworks.ai/inference/v1",
    api_key=os.environ["FIREWORKS_API_KEY"],
    timeout=120,
)

MAX_RETRIES = 3
RETRY_BASE_SEC = 1.5
DELAY_SEC = 1.2
FALLBACK_MODEL = None

def _probe_chat_model(mid: str):
    try:
        r = client.chat.completions.create(
            model=mid,
            messages=[{"role": "user", "content": 'Return only {"ok":true}'}],
            temperature=0,
            max_tokens=12,
        )
        _ = r.choices[0].message.content
        return True, "OK"
    except Exception as e:
        return False, str(e)

# 1) Try account-exposed models first
candidates = []
try:
    listed = [m.id for m in client.models.list().data]
    qwen_listed = [m for m in listed if "qwen" in m.lower()]
    # prefer instruct/chat-like ids first
    qwen_listed = sorted(qwen_listed, key=lambda x: (("instruct" not in x.lower()), x.lower()))
    candidates.extend(qwen_listed)
except Exception:
    pass

# 2) Fallback candidate ids (common Fireworks naming)
fallback_candidates = [
    "accounts/fireworks/models/qwen2p5-7b-instruct",
    "accounts/fireworks/models/qwen2p5-14b-instruct",
    "accounts/fireworks/models/qwen2p5-72b-instruct",
    "accounts/fireworks/models/qwen3-8b",
    "accounts/fireworks/models/qwen3-14b",
    "accounts/fireworks/models/qwen3-32b",
]
for m in fallback_candidates:
    if m not in candidates:
        candidates.append(m)

MODEL_NAME = None
probe_report = {}

for m in candidates:
    ok, msg = _probe_chat_model(m)
    probe_report[m] = msg[:160]
    if ok:
        MODEL_NAME = m
        break

if MODEL_NAME is None:
    raise RuntimeError(
        "No working Qwen model found on your Fireworks account.\n"
        f"Probe report: {probe_report}"
    )

print("MODEL_NAME:", MODEL_NAME)
print("Tried:", len(candidates), "models")


MODEL_NAME: accounts/fireworks/models/qwen3-8b
Tried: 6 models


In [12]:
from openai import OpenAI

MODEL_NAME = "accounts/fireworks/models/qwen2p5-7b-instruct"  # Qwen only
FALLBACK_MODEL = None

MAX_RETRIES = 3
RETRY_BASE_SEC = 1.5
DELAY_SEC = 1.2

client = OpenAI(
    base_url="https://api.fireworks.ai/inference/v1",
    api_key=os.environ["FIREWORKS_API_KEY"],
)

print("MODEL_NAME:", MODEL_NAME)


MODEL_NAME: accounts/fireworks/models/qwen2p5-7b-instruct


In [13]:
# Data + label loading

def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def load_master(master_path, test_rows):
    if not master_path:
        return test_rows
    with open(master_path, 'r', encoding='utf-8') as f:
        obj = json.load(f)
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict) and 'records' in obj and isinstance(obj['records'], list):
        return obj['records']
    return test_rows

test_records = load_jsonl(TEST_FILE)
master_records = load_master(MASTER_FILE, test_records)

VALID_CATEGORIES = sorted({str(r.get('category','')).strip() for r in master_records if r.get('category')})
VALID_SEVERITIES = sorted({str(r.get('severity','')).strip().lower() for r in master_records if r.get('severity')})
VALID_DEPARTMENTS = sorted({
    str(r.get('routing', {}).get('primary_department', '')).strip()
    for r in master_records
    if r.get('routing', {}).get('primary_department')
})

print(f'Loaded {len(test_records)} test records')
print('VALID_CATEGORIES :', VALID_CATEGORIES)
print('VALID_SEVERITIES :', VALID_SEVERITIES)
print('VALID_DEPARTMENTS:', VALID_DEPARTMENTS)


Loaded 32 test records
VALID_CATEGORIES : ['Broken Hospital Bed', 'Crowded Hospital Waiting Room', 'Dirty Hospital Bathroom', 'Empty / Unstaffed Nursing Station', 'Overflowing Hospital Trash (Outside)', 'Rats / Rodent Infestation', 'Torn Hospital Privacy Curtain', 'Unappetizing Hospital Food Tray', 'Unhygienic / Contaminated Hospital Food', 'Water Puddle on Hospital Floor']
VALID_SEVERITIES : ['critical', 'high', 'low', 'medium']
VALID_DEPARTMENTS: ['Administration', 'Dietary', 'Facilities Management', 'Housekeeping', 'Maintenance', 'Nursing', 'Pest Control', 'Waste Management']


In [14]:
# Prompts + parser + HF call

def build_zeroshot_prompt(caption, voice_text):
    return f"""A patient filed a hospital complaint.

Image caption: {caption}
Patient voice complaint: {voice_text}

Return ONLY JSON with keys: category, severity, department, complaint_description"""


def build_prompted_prompt(caption, voice_text):
    categories_str = "\n".join(f"- {c}" for c in VALID_CATEGORIES)
    departments_str = ", ".join(VALID_DEPARTMENTS)
    return f"""You are a hospital complaint triage system.
Return ONLY a valid JSON object (no markdown, no explanation) with keys:
category, severity, department, complaint_description

Allowed categories:
{categories_str}

Allowed severity:
- low
- medium
- high
- critical

Allowed departments:
{departments_str}

Example 1:
Input: bathroom mold and dirt + user says filthy bathroom
Output: {{"category":"Dirty Hospital Bathroom","severity":"high","department":"Housekeeping","complaint_description":"Patient reported severe unhygienic bathroom conditions including mold, dirt, and foul odor."}}

Example 2:
Input: rat near kitchen + user says rat seen near food area
Output: {{"category":"Rats / Rodent Infestation","severity":"critical","department":"Pest Control","complaint_description":"Patient observed rodent presence near food area, posing serious hygiene and infection risk."}}

Now process:
Image caption: {caption}
Voice complaint: {voice_text}
"""


def parse_json_output(raw_text):
    if not raw_text:
        return {}, False
    cleaned = re.sub(r"```(?:json)?", "", str(raw_text), flags=re.IGNORECASE).replace("```", "").strip()
    s = cleaned.find('{'); e = cleaned.rfind('}')
    if s != -1 and e != -1 and e > s:
        try:
            return json.loads(cleaned[s:e+1]), True
        except Exception:
            pass
    try:
        return json.loads(cleaned), True
    except Exception:
        return {}, False


def extract_fields(parsed):
    return (
        str(parsed.get('category', '')).strip(),
        str(parsed.get('severity', '')).strip().lower(),
        str(parsed.get('department', '')).strip(),
        str(parsed.get('complaint_description', '')).strip(),
    )


STOP_RUN_DUE_TO_BILLING = False

def call_model(prompt, record_id, baseline_name):
    global STOP_RUN_DUE_TO_BILLING
    if STOP_RUN_DUE_TO_BILLING:
        return None

    messages = [
        {"role": "system", "content": "Return ONLY valid JSON."},
        {"role": "user", "content": prompt},
    ]

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages,
                temperature=0,
                max_tokens=512,
            )
            content = resp.choices[0].message.content
            return (content or "").strip()

        except Exception as e:
            last_err = str(e)

            # billing hard-stop
            if "402" in last_err or "Payment Required" in last_err:
                STOP_RUN_DUE_TO_BILLING = True
                print("\n[STOP] Billing/credit limit hit. Stopping run.")
                return None

            # retry transient only
            if any(k in last_err.lower() for k in ["429","rate","timeout","503","502","504"]) and attempt < MAX_RETRIES:
                time.sleep(RETRY_BASE_SEC * (2 ** (attempt - 1)))
                continue
            break

    print(f"  ERROR on {record_id} [{baseline_name}]: {last_err}")
    return None


In [15]:
# Runner + metrics + save

def compute_metrics(results, baseline_name):
    gt_cats = [r['gt_category'] for r in results]
    gt_sevs = [r['gt_severity'] for r in results]
    gt_deps = [r['gt_department'] for r in results]
    pr_cats = [r['pred_category'] for r in results]
    pr_sevs = [r['pred_severity'] for r in results]
    pr_deps = [r['pred_department'] for r in results]

    json_valid_count = sum(1 for r in results if r['json_valid'])

    cat_acc = accuracy_score(gt_cats, pr_cats)
    sev_acc = accuracy_score(gt_sevs, pr_sevs)
    dep_acc = accuracy_score(gt_deps, pr_deps)
    cat_f1 = f1_score(gt_cats, pr_cats, average='macro', zero_division=0)
    sev_f1 = f1_score(gt_sevs, pr_sevs, average='macro', zero_division=0)
    dep_f1 = f1_score(gt_deps, pr_deps, average='macro', zero_division=0)

    invalid_cats = sum(1 for x in pr_cats if x not in VALID_CATEGORIES)
    invalid_sevs = sum(1 for x in pr_sevs if x not in VALID_SEVERITIES)
    invalid_deps = sum(1 for x in pr_deps if x not in VALID_DEPARTMENTS)

    smoother = SmoothingFunction().method1
    scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

    bleu_scores=[]; r1=[]; r2=[]; rl=[]
    for r in results:
        ref = r['gt_description'].split(); hyp = r['pred_description'].split()
        if ref and hyp:
            bleu_scores.append(sentence_bleu([ref], hyp, smoothing_function=smoother))
            rs = scorer.score(r['gt_description'], r['pred_description'])
            r1.append(rs['rouge1'].fmeasure); r2.append(rs['rouge2'].fmeasure); rl.append(rs['rougeL'].fmeasure)

    def avg(v):
        return round(sum(v)/len(v), 4) if v else 0.0

    print(f"\nPer-class CATEGORY report [{baseline_name}]")
    print(classification_report(gt_cats, pr_cats, zero_division=0))

    return {
        'baseline': baseline_name,
        'model': MODEL_NAME,
        'total_records': len(results),
        'json_validity_rate': round(json_valid_count / len(results), 4),
        'category_accuracy': round(cat_acc, 4),
        'category_macro_f1': round(cat_f1, 4),
        'severity_accuracy': round(sev_acc, 4),
        'severity_macro_f1': round(sev_f1, 4),
        'department_accuracy': round(dep_acc, 4),
        'department_macro_f1': round(dep_f1, 4),
        'bleu_score': avg(bleu_scores),
        'rouge1': avg(r1),
        'rouge2': avg(r2),
        'rougeL': avg(rl),
        'invalid_category_preds': invalid_cats,
        'invalid_severity_preds': invalid_sevs,
        'invalid_department_preds': invalid_deps,
    }


def run_baseline(rows, baseline_name, prompt_fn):
    print('\n' + '='*60)
    print(f'Running: {baseline_name} ({len(rows)} records)')
    print('='*60)
    out=[]
    ckpt_path = os.path.join(OUTPUT_DIR, f"checkpoint_{baseline_name}.csv")

    for i, rec in enumerate(rows, 1):
        image_id = rec.get('image_id', f'row_{i}')
        caption = rec.get('input', {}).get('image_caption', rec.get('refined_caption', ''))
        voice_text = rec.get('voice_text', '')
        gt_cat = rec.get('category', '')
        gt_sev = str(rec.get('severity', '')).lower()
        gt_dep = rec.get('routing', {}).get('primary_department', rec.get('department', ''))
        gt_desc = rec.get('complaint_description', '')

        print(f"  [{i:02d}/{len(rows)}] {image_id} ... ", end='', flush=True)
        raw = call_model(prompt_fn(caption, voice_text), image_id, baseline_name)
        parsed, jv = parse_json_output(raw)
        pc, ps, pdp, pdesc = extract_fields(parsed)

        cat_ok = (pc == gt_cat)
        sev_ok = (ps == gt_sev)
        dep_ok = (pdp == gt_dep)

        print(('cat✓' if cat_ok else 'cat✗'), '|', ('sev✓' if sev_ok else 'sev✗'), '|', ('json✓' if jv else 'json✗'))

        out.append({
            'baseline': baseline_name,
            'image_id': image_id,
            'gt_category': gt_cat,
            'gt_severity': gt_sev,
            'gt_department': gt_dep,
            'gt_description': gt_desc,
            'pred_category': pc,
            'pred_severity': ps,
            'pred_department': pdp,
            'pred_description': pdesc,
            'json_valid': jv,
            'raw_output': raw or '',
            'cat_correct': cat_ok,
            'sev_correct': sev_ok,
            'dept_correct': dep_ok,
        })

        pd.DataFrame(out).to_csv(ckpt_path, index=False)
        time.sleep(DELAY_SEC)

    return out

# Optional smoke test
smoke = build_zeroshot_prompt('A broken hospital bed', 'My bed is broken and unsafe')
print('\nSmoke test output:')
print(call_model(smoke, 'smoke_1', 'SmokeTest'))

# Full run
results_zs = run_baseline(test_records, 'Baseline-1-ZeroShot', build_zeroshot_prompt)
metrics_zs = compute_metrics(results_zs, 'Baseline-1-ZeroShot')
results_pt = run_baseline(test_records, 'Baseline-2-Prompted', build_prompted_prompt)
metrics_pt = compute_metrics(results_pt, 'Baseline-2-Prompted')

all_results = results_zs + results_pt
all_metrics = [metrics_zs, metrics_pt]

results_csv = os.path.join(OUTPUT_DIR, 'baseline_results.csv')
metrics_json = os.path.join(OUTPUT_DIR, 'baseline_metrics.json')
errors_csv = os.path.join(OUTPUT_DIR, 'baseline_errors.csv')

pd.DataFrame(all_results).to_csv(results_csv, index=False)
with open(metrics_json, 'w', encoding='utf-8') as f:
    json.dump(all_metrics, f, indent=2)

errors = [r for r in all_results if (not r['cat_correct']) or (not r['sev_correct']) or (not r['json_valid'])]
pd.DataFrame(errors).to_csv(errors_csv, index=False)

print('\nSaved:')
print(' -', results_csv)
print(' -', metrics_json)
print(' -', errors_csv)
print('\nSummary:')
print(pd.DataFrame(all_metrics)[[
    'baseline','model','json_validity_rate',
    'category_accuracy','category_macro_f1',
    'severity_accuracy','severity_macro_f1',
    'department_accuracy','department_macro_f1',
    'bleu_score','rouge1','rouge2','rougeL'
]])



Smoke test output:
  ERROR on smoke_1 [SmokeTest]: Error code: 404 - {'error': {'message': 'Model not found, inaccessible, and/or not deployed', 'param': 'model', 'code': 'NOT_FOUND', 'type': 'error'}, 'request_id': 'chatcmpl-15cfd6aa3c1f4b28a843390536f4726a'}
None

Running: Baseline-1-ZeroShot (32 records)
  [01/32] broken_9 ...   ERROR on broken_9 [Baseline-1-ZeroShot]: Error code: 404 - {'error': {'message': 'Model not found, inaccessible, and/or not deployed', 'param': 'model', 'code': 'NOT_FOUND', 'type': 'error'}, 'request_id': 'chatcmpl-d1ccb37c1d924b9a9b6076b417fb438d'}
cat✗ | sev✗ | json✗
  [02/32] broken_1 ...   ERROR on broken_1 [Baseline-1-ZeroShot]: Error code: 404 - {'error': {'message': 'Model not found, inaccessible, and/or not deployed', 'param': 'model', 'code': 'NOT_FOUND', 'type': 'error'}, 'request_id': 'chatcmpl-e49c965b33164be9961112f5aac4dca2'}
cat✗ | sev✗ | json✗
  [03/32] broken_4 ...   ERROR on broken_4 [Baseline-1-ZeroShot]: Error code: 404 - {'error': {'me

KeyboardInterrupt: 

In [ ]:
# Colab: download all generated output files in one ZIP
import os
import zipfile
from google.colab import files

output_dir = "/content/phase2_baseline_outputs"
zip_path = "/content/phase2_baseline_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(output_dir):
        fpath = os.path.join(output_dir, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, arcname=fname)

print("Created:", zip_path)
files.download(zip_path)


Created: /content/phase2_baseline_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>